## Basic Usage

Input

In [ ]:
from phytclust import PhytClust

# Pass a Biopylo Tree
pc1 = PhytClust(tree_obj)

# Pass a Newick file path (str)
pc2 = PhytClust("tree.nwk")

# Pass a Newick file path (Path)
from pathlib import Path

pc3 = PhytClust(Path("tree.nwk"))

# Pass an inline Newick string
pc4 = PhytClust("((A,B),C);")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")

from Bio import Phylo
from phytclust import PhytClust

tree = Phylo.read(
    "/home/ganesank/project/phytclust_additional_files/data/real_trees/nextstrain_dengue_all_genome_tree.nwk",
    "newick",
)

# Basic instance: no outgroup, no constraints
pc = PhytClust(tree, optimize_polytomies=False, polytomy_mode="hard")

# # 1) Exact k = 5 clusters
# result_k5 = pc.run(k=5)
# clusters_k5 = result_k5["clusters"]

# 2) Global solutions, top 3 peaks
result_global = pc.run(top_n=3, max_k=round(tree.count_terminals()*0.6))
peaks = result_global["peaks"]  # e.g. [4, 7, 12]
clusterings = result_global["clusters"]

# # 3) Multi-resolution: one peak per log-bin
# result_res = pc.run(by_resolution=True, num_bins=3)
# ks = result_res["ks"]
# clusterings_res = result_res["clusters"]  # clusters are always dicts: Clade -> int cluster ID

In [ ]:
result_global["peaks"]

In [ ]:
pc.plot(k=9)

In [ ]:
pc.plot(k=7, height_scale=0.4)

## Outgroup and minimum cluster size

In [ ]:
# Exclude one taxon as outgroup, require final clusters to have ≥ 3 leaves
pc = PhytClust(tree, outgroup="Homo_sapiens", min_cluster_size=3)
result = pc.run(k=6)
clusters = result["clusters"]

- The outgroup is removed from the terminal set and does not appear in any cluster.

- min_cluster_size is a hard constraint: if no valid partition into k clusters
exists under this constraint, get_clusters / run(k=...) raises a ValueError.

## Penalising small “outlier” clusters (soft)

If you want to discourage very small clusters rather than forbid them:

In [ ]:
pc = PhytClust(tree, min_cluster_size=1)

# Treat clusters with < 3 leaves as outliers
pc.outlier_size_threshold = 3
pc.outlier_penalty = 1000.0  # large additive penalty on their cost

res = pc.run(k=6)
clusters = res["clusters"]

This does not change feasibility, only costs:

- small clusters are still allowed,
- but the DP will avoid them if larger, better-supported clusters are possible.

## Using branch support

Branch support can be folded into the clustering objective:

In [ ]:
pc = PhytClust(
    "/home/ganesank/project/phytclust/doc/example/sample_bootstrap.newick",
    use_branch_support=False,
    min_support=0.05,  # lower bound on support (in 0–1)
    support_weight=1.0,  # how strongly to penalise low-support edges
)

res = pc.run(k=3)
# clusters = res["clusters"][0]

Visualization and Saving

## Extra utilities

PhytClust contains a few helper metrics and selection functions:

In [ ]:
from Bio import Phylo
from phytclust.metrics import normalized_colless, calculate_total_length_variation
from phytclust.selection import select_representative_species

tree = Phylo.read("tree.nwk", "newick")

nc = normalized_colless(tree)
rt_tip_sd = calculate_total_length_variation(tree)

pc = PhytClust(tree)
res = pc.run(k=6)
clusters = res["clusters"]

reps = select_representative_species(
    tree,
    clusters,
    mode="maximize",
    distance_ref="mrca",
)

## Calculating bootstrap stability scores

In [ ]:
from Bio import Phylo
from phytclust.algo.bootstrap_stability import choose_k_by_stability

trees = list(Phylo.parse("my_bootstrap_trees.newick", "newick"))

result = choose_k_by_stability(
    trees,
    k_values=range(2, 11),      outgroup="diploid",
    min_cluster_size=1,
    pc_kwargs={"use_branch_support": False},)

print("Best k by stability:", result["best_k"])
print("All scores:", result["scores"])
coassoc_best = result["coassoc"][result["best_k"]]
taxa = result["taxa"]

---------------------------
